# HW: Semantic Image Segmentation with U-Net (Oxford-IIIT Pets)

## Deep Learning / Computer Vision — Programming Assignment

**Big picture.** Semantic segmentation labels **every pixel**. You will build,
train and evaluate a **U-Net** with a **pretrained MobileNetV2 encoder** (transfer
learning) that outputs a per-pixel class map for pet images.

**Notebook input →** the **Oxford-IIIT Pet** dataset (ships with TensorFlow
Datasets — no download needed). Each mask pixel is one of 3 classes
(`1=pet, 2=border, 3=background`; you shift these to `0,1,2`).
**Notebook output →** a trained segmentation model, loss curves, mean-IoU, and
predicted masks.

Architecture (fill in the wiring yourself):
```
Input 128x128x3 → [frozen MobileNetV2 encoder, 5 skip taps] → [upsampling decoder + skip concat] → 128x128x3 logits
```

You are expected to **read the Keras / tf.data docs and work out the details
yourself.** Cells state only the input/output of each function.

### Deliverables & grading (100 pts)
| Part | Topic | Pts |
|------|-------|-----|
| 1 | Load & explore | 10 |
| 2 | Data pipeline | 15 |
| 3 | Visualization helper | 5 |
| 4 | Encoder (down_stack) | 10 |
| 5 | Decoder + U-Net wiring | 25 |
| 6 | Compile (loss/metrics) | 10 |
| 7 | Train + per-epoch preview | 10 |
| 8 | Evaluation (loss curves + mean IoU) | 10 |
| — | Written answers (✍️) | 5 |

> Fill in every `# TODO` and ✍️ question. Keep the provided asserts and headers.
> A GPU runtime is strongly recommended.


## Part 0 — Setup  *(provided)*

In [ ]:
!pip install -q git+https://github.com/tensorflow/examples.git

In [ ]:
import tensorflow as tf
from tensorflow_examples.models.pix2pix import pix2pix
import tensorflow_datasets as tfds
from IPython.display import clear_output
import matplotlib.pyplot as plt
import numpy as np
print("TF", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices('GPU')))


## Part 1 — Load & explore  *(10 pts)*

In [ ]:
# TODO: load 'oxford_iiit_pet:3.*.*' with info into (dataset, info).
dataset, info = tfds.load('oxford_iiit_pet:4.*.*', with_info=True)

assert dataset is not None and 'train' in dataset and 'test' in dataset
print("train:", info.splits['train'].num_examples, "test:", info.splits['test'].num_examples)


In [ ]:
# TODO: take one raw training example; print image/mask shapes and value ranges.
# Confirm the mask values are in {1,2,3}.

example = next(iter(dataset['train'].take(1)))
image, mask = example['image'], example['segmentation_mask']

print("image shape:", image.shape, "dtype:", image.dtype,"range:", (int(tf.reduce_min(image)), int(tf.reduce_max(image))))
print("mask shape:", mask.shape, "dtype:", mask.dtype,"range:", (int(tf.reduce_min(mask)), int(tf.reduce_max(mask))))
print("unique mask values:", np.unique(mask.numpy()))



✍️ **Q1.1** Why must labels start at 0 for `SparseCategoricalCrossentropy`,
and what do the three classes mean?

**Your answer:** _SparseCategoricalCrossentropy treats each integer label as an index into the model's output channel axis. Decoder's final layer outputs 3 channels, indexed 0, 1, 2. If labels stay at 1, 2, 3, label 3 has no matching channel, and even where it doesn't crash outright, the loss would be comparing each pixel against the wrong channel. Shifting to 0, 1, 2 makes each label line up exactly with the channel meant to predict it._

## Part 2 — Data pipeline  *(15 pts)*  — all images become 128×128.

In [ ]:
# TODO: normalize(input_image, input_mask) -> (image scaled to [0,1] float,
# mask shifted to {0,1,2}).
def normalize(input_image, input_mask):
    input_image = tf.cast(input_image, tf.float32) / 255.0
    input_mask -= 1
    return input_image, input_mask


In [ ]:
# TODO: load_image_train(datapoint) -> (image, mask), resized to 128x128, with
# random left-right flip applied to BOTH consistently, then normalized.
@tf.function
def load_image_train(datapoint):
    input_image = tf.image.resize(datapoint['image'], (128, 128))
    input_mask = tf.image.resize(datapoint['segmentation_mask'], (128, 128), method='nearest')

    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        input_mask = tf.image.flip_left_right(input_mask)

    input_image, input_mask = normalize(input_image, input_mask)
    
    return input_image, input_mask

# TODO: load_image_test(datapoint) -> (image, mask), same but no augmentation.
def load_image_test(datapoint):
    input_image = tf.image.resize(datapoint['image'], (128, 128))
    input_mask = tf.image.resize(datapoint['segmentation_mask'], (128, 128),method='nearest')

    input_image, input_mask = normalize(input_image, input_mask)
    
    return input_image, input_mask


✍️ **Q2.1** What breaks if you flip the image but not the mask?

**Your answer:** _If we flip the image but leave the mask unflipped, the two tensors become spatially misaligned — pixel (i, j) in the image no longer corresponds to pixel (i, j) in the mask. Concretely, if a cat's head was on the left side of the original image, flipping the image moves it to the right side, but the mask still marks "pet" pixels on the left. The model would then be trained to associate the background region under the flipped head with the pet label.
Loss would still go down somewhat during training (the model partially learns from the unflipped half of each batch on average), but IoU and visual predictions would come out worse, with segmentation boundaries that look shifted or mirrored relative to the actual object.

In [ ]:
# TODO: build the batched pipelines.
TRAIN_LENGTH = info.splits['train'].num_examples
BATCH_SIZE, BUFFER_SIZE = 64, 1000
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

train =  dataset['train'].map(load_image_train, num_parallel_calls=tf.data.AUTOTUNE) # dataset['train'] mapped through load_image_train
test = dataset['test'].map(load_image_test, num_parallel_calls=tf.data.AUTOTUNE)     # dataset['test']  mapped through load_image_test
train_dataset = (
    train
    .cache()
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)   # cached, shuffled, batched, repeated, prefetched
test_dataset = test.batch(BATCH_SIZE)     # batched

assert train_dataset is not None and test_dataset is not None
for img, msk in train.take(1):
    assert img.shape == (128, 128, 3), img.shape
    assert float(tf.reduce_max(msk)) <= 2, "mask classes must be in {0,1,2}"
print("pipeline ok")


## Part 3 — Visualization helper  *(5 pts)*

In [ ]:
# TODO: display(display_list) -> plots a row of images with titles
# ['Input Image','True Mask','Predicted Mask'] (as many as given).
def display(display_list):
    titles = ['Input Image', 'True Mask', 'Predicted Mask']
    plt.figure(figsize=(15, 15))
    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i + 1)
        plt.title(titles[i])
        plt.imshow(tf.keras.utils.array_to_img(display_list[i]))
        plt.axis('off')
    plt.show()

for image, mask in train.take(1):
    sample_image, sample_mask = image, mask
display([sample_image, sample_mask])


## Part 4 — Encoder  *(10 pts)*  — reuse a pretrained MobileNetV2, tap 5 maps.

In [ ]:
# TODO: build `down_stack` = a Keras model from MobileNetV2(input [128,128,3],
# include_top=False) whose outputs are these 5 layers, and FREEZE it.
LAYER_NAMES = ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu',
               'block_13_expand_relu', 'block_16_project']  # -> 64,32,16,8,4

base_model = tf.keras.applications.MobileNetV2(input_shape=[128, 128, 3],include_top=False)
layers = [base_model.get_layer(name).output for name in LAYER_NAMES]

down_stack = tf.keras.Model(inputs=base_model.input, outputs=layers)
down_stack.trainable = False

assert down_stack is not None and down_stack.trainable is False
print("encoder outputs:", [o.shape[1:3] for o in down_stack.outputs])


✍️ **Q4.1** Why freeze the pretrained encoder? When would you unfreeze it?

**Your answer:** _Freezing keeps MobileNetV2's ImageNet-pretrained weights fixed since Oxford-IIIT Pets is too small to relearn good general-purpose features without overfitting or destroying them, and it prevents the randomly-initialized decoder's large early gradients from wrecking those weights before the decoder learns anything useful — it also cuts training cost by skipping gradients/optimizer state for the encoder. We'd unfreeze (usually just the last few blocks) after the decoder converges, using a much smaller learning rate to fine-tune features toward pet segmentation specifically._

## Part 5 — Decoder + U-Net wiring  *(25 pts, core)*

The decoder upsamples the bottleneck back to 128×128, **concatenating** each
upsampled map with the matching encoder skip — that is what makes it a U-Net.

In [ ]:
# Provided: upsampling blocks (mirror the encoder resolutions).
up_stack = [pix2pix.upsample(512, 3), pix2pix.upsample(256, 3),
            pix2pix.upsample(128, 3), pix2pix.upsample(64, 3)]


In [ ]:
def unet_model(output_channels):
    """Input: Keras input 128x128x3. Output: a tf.keras.Model producing
    128x128x`output_channels` logits, wiring the frozen encoder to the decoder
    with skip connections."""
    inputs = tf.keras.layers.Input(shape=[128, 128, 3])

    skips = down_stack(inputs)
    x = skips[-1]                 
    skips = reversed(skips[:-1])  

    for up, skip in zip(up_stack, skips):
        x = up(x)                                        
        x = tf.keras.layers.Concatenate()([x, skip])     

    last = tf.keras.layers.Conv2DTranspose(
        filters=output_channels, kernel_size=3, strides=2, padding='same')
    x = last(x)

    return tf.keras.Model(inputs=inputs, outputs=x)

OUTPUT_CHANNELS = 3
model = unet_model(OUTPUT_CHANNELS)
assert model.output_shape[1:] == (128, 128, 3), model.output_shape
print("U-Net output:", model.output_shape)


In [ ]:
try:
    tf.keras.utils.plot_model(model, show_shapes=True)
except Exception as e:
    print("plot_model skipped:", e); model.summary()


✍️ **Q5.1** What do the skip connections do, and why would a no-skip
encoder→decoder give blurrier masks?

**Your answer:** _Skip connections pass encoder feature maps directly to the decoder at matching resolutions, giving it sharp local edge detail alongside the bottleneck's coarse "what." Without skips, all spatial detail must survive compression to a 4×4 bottleneck and be reconstructed by upsampling alone, which can only interpolate, not invent lost detail. This makes boundaries blur and round off instead of following true object edges._

## Part 6 — Compile  *(10 pts)*  — the last layer outputs raw logits.

In [ ]:
# TODO: compile `model` with a suitable optimizer, a loss consistent with raw
# (non-softmax) logits and integer masks, and an accuracy metric.

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


✍️ **Q6.1** Why `from_logits=True`?
✍️ **Q6.2** Why the *sparse* categorical crossentropy given the mask shape?

**Your answers:**

- Q6.1: _from_logits=True tells the loss to apply softmax internally, since decoder's last layer has no activation and outputs raw logits, not probabilities. Combining softmax + crossentropy in one op is also more numerically stable than doing them separately._
  
- Q6.2: _Masks hold integer class indices (0,1,2) with shape (128,128,1), not one-hot vectors of shape (128,128,3). Sparse categorical crossentropy accepts integer labels directly, avoiding the need to one-hot-encode every mask._

## Part 7 — Prediction helpers + training  *(10 pts)*

In [ ]:
# TODO: create_mask(pred_mask) -> a single 2D label map (argmax over classes,
# first item of the batch, trailing channel axis).
def create_mask(pred_mask):
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    return pred_mask[0].numpy()

# TODO: show_predictions(dataset=None, num=1) -> displays image, true mask, and
# predicted mask (from `dataset` if given, else from the sample image).
def show_predictions(dataset=None, num=1):
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = model.predict(image)
            display([image[0], mask[0], create_mask(pred_mask)])
    else:
        pred_mask = model.predict(sample_image[tf.newaxis, ...])
        display([sample_image, sample_mask, create_mask(pred_mask)])

show_predictions()


In [ ]:
# Provided: preview a prediction after every epoch.
class DisplayCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True); show_predictions()
        print(f"\nSample Prediction after epoch {epoch + 1}\n")

# TODO: train the model -> store the history in `model_history`.
EPOCHS, VAL_SUBSPLITS = 20, 5
VALIDATION_STEPS = info.splits['test'].num_examples // BATCH_SIZE // VAL_SUBSPLITS
model_history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    validation_data=test_dataset,
    callbacks=[DisplayCallback()]
)


## Part 8 — Evaluation  *(10 pts)*  — accuracy is weak here; also report IoU.

In [ ]:
# TODO: plot training vs validation loss from model_history.history.

loss = model_history.history['loss']
val_loss = model_history.history['val_loss']
epochs_range = range(len(loss))

plt.figure()
plt.plot(epochs_range, loss, 'r', label='Training loss')
plt.plot(epochs_range, val_loss, 'bo', label='Validation loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss Value')
plt.legend()
plt.show()


In [ ]:
# TODO: compute MEAN IoU on the test set with tf.keras.metrics.MeanIoU(num_classes=3).
miou = tf.keras.metrics.MeanIoU(num_classes=3)

for image_batch, mask_batch in test_dataset:
    pred_batch = model.predict(image_batch, verbose=0)
    pred_batch = tf.argmax(pred_batch, axis=-1)[..., tf.newaxis]
    miou.update_state(mask_batch, pred_batch)

print("Mean IoU:", float(miou.result()))


In [ ]:
# TODO: show 3 predictions from the test set.
show_predictions(test_dataset, num=3)

✍️ **Q8.1 (Reflection)** Compare final accuracy vs mean IoU. Which class is
hardest and why? Name two changes to improve IoU and why each helps.

### Bonus (+10)
- Dice / focal loss vs the baseline.
- Unfreeze top MobileNetV2 blocks and fine-tune.
- Per-class IoU, discussing the border class.

**Your answer / bonus notes:** _Mean IoU will come in noticeably lower than pixel accuracy (e.g. ~0.90 accuracy but IoU in the 0.7 range) because accuracy is dominated by the easy, majority background class, while IoU averages equally across all three classes regardless of size.
The border class is hardest — it's a thin ring of pixels with high inherent label ambiguity, so small localization errors cost it disproportionately in IoU. 

Two changes to improve IoU: 
(1) fine-tune (unfreeze) the top encoder blocks at a low learning rate — lets features adapt specifically to pet-boundary detail rather than generic ImageNet features

(2) We can use a Dice or focal loss and directly optimizes for class overlap and up-weights the hard/minority border pixels that plain crossentropy under-prioritizes._

In [ ]:
# Bonus: Dice / focal loss vs the baseline.

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_true_oh = tf.one_hot(y_true, depth=OUTPUT_CHANNELS)         
    y_pred_soft = tf.nn.softmax(y_pred, axis=-1)                   
    
    intersection = tf.reduce_sum(y_true_oh * y_pred_soft, axis=[1, 2])
    union = tf.reduce_sum(y_true_oh, axis=[1, 2]) + tf.reduce_sum(y_pred_soft, axis=[1, 2])
    dice_per_class = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(dice_per_class)


def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    y_true_oh = tf.one_hot(y_true, depth=OUTPUT_CHANNELS)
    y_pred_soft = tf.nn.softmax(y_pred, axis=-1)
    y_pred_soft = tf.clip_by_value(y_pred_soft, 1e-8, 1.0 - 1e-8)

    ce = -y_true_oh * tf.math.log(y_pred_soft)
    weight = alpha * tf.pow(1.0 - y_pred_soft, gamma)
    return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))


def ce_dice_loss(y_true, y_pred):
    ce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)(y_true, y_pred)
    return ce + dice_loss(y_true, y_pred)


BONUS_EPOCHS = 10

def build_and_train(loss_fn, name):
    m = unet_model(OUTPUT_CHANNELS)         
    m.compile(optimizer='adam', loss=loss_fn, metrics=['accuracy'])
    print(f"--- training with {name} ---")
    h = m.fit(train_dataset, epochs=BONUS_EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
              validation_data=test_dataset, validation_steps=VALIDATION_STEPS, verbose=0)
    return m, h

dice_model, dice_hist = build_and_train(dice_loss, "Dice loss")
focal_model, focal_hist = build_and_train(focal_loss, "Focal loss")
combo_model, combo_hist = build_and_train(ce_dice_loss, "CE + Dice")

def compute_miou(m):
    miou = tf.keras.metrics.MeanIoU(num_classes=3)
    for image_batch, mask_batch in test_dataset:
        pred_batch = m.predict(image_batch, verbose=0)
        pred_batch = tf.argmax(pred_batch, axis=-1)[..., tf.newaxis]
        miou.update_state(mask_batch, pred_batch)
    return float(miou.result())

results = {
    "CE (baseline)": compute_miou(model),
    "Dice": compute_miou(dice_model),
    "Focal": compute_miou(focal_model),
    "CE + Dice": compute_miou(combo_model),
}
for name, iou in results.items():
    print(f"{name:15s} Mean IoU: {iou:.4f}")